<h1 style="font-family: 'Times New Roman', Times, serif; text-align: center; font-size: 40px;">
    M2PF-Data-Challenge
</h1>

<p style="font-family: 'Times New Roman', Times, serif; text-align: center; font-size: 20px;">
    Michiel Nys
</p>

# 0. Idea

Train a model to generate the next plausible return, given previous returns, volumes and some categorical data.

### Following data is available

- **$TS$:** Anonymized and shuffled timestamp of data
- **$ALLOCATION$:** Label for specific allocation (i.e., different trading strategies)
- **$RET_i$:** Allocation's return at time $i = 1,2,...,20$
- **$SIGNED\_VOLUME_i$:** Allocation's signed volume at time $i = 1,2,...,20$
- **$MEDIAN\_DAILY\_TURNOVER$:** Allocation's median daily turnover, over the 20 fixings
- **$GROUP$:** Anonymized allocation's group (ie. long-short, momentum, ...)
- **$TARGET$:** Return at time 21

### Structure

The neural network consists of two components:

1. **Transformer:** A neural network architecture that uses self-attention to model dependencies between different time steps. In this case, each time step contains both $RET_i$ and $SIGNED\_VOLUME_i$.

2. **Embedding:** Learnable embedding layers that map categorical variables (e.g. group and allocation) to continuous vector representations.

The outputs of the two components are then concatenated and passed to a final layer to make the prediction. A diagram to visualize this:

<figure style="text-align: center; max-width: 70%; margin: 0 auto;">
    <img src="../resources/model_architecture.png"
         alt="Model Design"
         style="width: 100%; height: auto; display: block;">
    <figcaption style="font-size: 16px; color: #555; margin-top: 5px;">
        Figure: Model Design
    </figcaption>
</figure>


# 1. Data

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load and prepare

In [ ]:
RET_COLS = [f"RET_{i+1}" for i in range(19, -1, -1)]
VOL_COLS = [f"SIGNED_VOLUME_{i+1}" for i in range(19, -1, -1)]

CAT_COLS = ["ALLOCATION", "GROUP"]
CONT_COLS = [
    "MEDIAN_DAILY_TURNOVER",
    "RET_MEAN",
    "RET_STD",
    "VOL_MEAN",
    "VOL_STD"
]

def prep_features(df):

    df = df.drop(columns=["ROW_ID", "TS"])

    # categorical variables
    df["ALLOCATION"] = (
        df["ALLOCATION"]
        .str.extract(r"(\d+)")
        .astype(int)
        .squeeze()
        - 1
    )

    df["GROUP"] = df["GROUP"].astype(int) - 1

    df[RET_COLS] = (
        df[RET_COLS]
        .interpolate(axis=1)
        .bfill(axis=1)
        .ffill(axis=1)
    )

    df[VOL_COLS] = (
        df[VOL_COLS]
        .interpolate(axis=1)
        .bfill(axis=1)
        .ffill(axis=1)
    )

    return df


def prep_targets(df):
    df["label"] = (df["target"] > 0).astype(int)
    return df


X_train = prep_features(pd.read_csv("/content/drive/MyDrive/Colab Notebooks/M2PF-Data-Challenge/X_train.csv"))
y_train = prep_targets(pd.read_csv("/content/drive/MyDrive/Colab Notebooks/M2PF-Data-Challenge/y_train.csv"))

# 2. Model

## Component A: Encoder

The time series data are combined into a (batch x time x channels):
- **batch**: the different independent samples
- **time**: the time steps in the series (default is 20)
- **channel**: the different features that are measured

The latter can be understood more intuitively by relating it to colour channels. In a picture for every coordinate (b x t) you have a combination of RGB values to obtain a pixel. Channel length is here 3 (red, green blue). In this case, the **channels represent different things measured on a particular day for a particular allocation**. Examples are:

- Standardized return
- Standardized volatility
- Performance compared to peers
- ...

Basically all things you can measure on a day-to-day basis

In [ ]:
import numpy as np

def standardize(x, axis=1):
    x_mean = x.mean(dim=axis, keepdim=True)
    x_std = x.std(dim=axis, keepdim=True)

    x_std = torch.where(x_std == 0, torch.ones_like(x_std), x_std)

    x = (x - x_mean) / x_std

    return x, x_mean, x_std


def build_channels(X_RET, X_VOL):
    """
    X_RET: np.array
        array of return: n_samples x n_steps
    X_vol: np.array
        array of vol: n_samples x n_steps
    """
    ch1, ret_mean, ret_std = standardize(X_RET, axis=1)
    ch2, vol_mean, vol_std = standardize(X_VOL, axis=1)

    channels = torch.stack([ch1, ch2], dim=-1) # (B, T, C)
    row_stats = torch.cat([ret_mean, ret_std, vol_mean, vol_std], dim=1)  # (B, V) # summarizing statistics on sample level (B, V) where V = # statistics

    return torch.clip(channels, -8, 8), row_stats

X_RET = torch.from_numpy(X_train[RET_COLS].to_numpy()).float()
X_VOL = torch.from_numpy(X_train[VOL_COLS].to_numpy()).float()
channels, row_stats = build_channels(X_RET, X_VOL)
print(row_stats.shape)

torch.Size([527073, 4])


### SequenceEncoder

The `SequenceEncoder` class is the heart of the first component. The model flow is as
follows:

**1. Channel expansion**

The `n_channels` are expanded to `d_model` to give room for capturing different
relationships. Very similar to word embeddings, the weights are trained and hard to
interpret.

**2. Positional embedding**

A positional embedding is created to take into account the order of the sequence
(day 20 → day 19 → ...).

**3. Transformer encoder layer**

A transformer encoder layer with `n_heads` self-attention blocks. This layer lets all
the days "talk to each other" and encode important relationships. There is no future
masking: day 8 is allowed to use information from day 3.

**4. Pooling**

Two things happen and get concatenated:

- **Attention pooling:** the model learns a weight for each of the 20 days and takes a
  weighted average. A quiet day gets little weight, a big move gets more.
- **The final timestep:** yesterday's vector, taken as-is.

### Component A head



In [ ]:
class SequenceEncoder(nn.Module):
    def __init__(self, n_channels=2, d_model=64, n_heads=4, n_layers=2, dropout=0.25, max_len=20):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(n_channels, d_model),
            nn.LayerNorm(d_model)
        )

        self.pos = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model = d_model,
            nhead = n_heads,
            dim_feedforward = d_model * 2,
            dropout = dropout,
            activation = "gelu",
            batch_first = True,
            norm_first = True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers = n_layers, enable_nested_tensor=False)
        self.final_ln = nn.LayerNorm(d_model)
        self.attn_score = nn.Linear(d_model, 1)
        self.out_dim = 2 * d_model

    def forward(self, x, pad_mask = None):
        T = x.size(1)
        h = self.proj(x) + self.pos[:, :T]
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        h = self.final_ln(h)

        scores = self.attn_score(h)
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask.unsqueeze(-1), float("-inf"))
        w = torch.softmax(scores, dim=1)

        pooled = (w * h).sum(dim=1)
        last = h[:, -1]
        return torch.cat([pooled, last], dim=-1)

class ComponentA(nn.Module):
    def __init__(self, n_channels=2, d_model=64, dropout=0.25):
        super().__init__()
        self.enc = SequenceEncoder(n_channels, d_model, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(self.enc.out_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x, pad_mask=None):
        return self.head(self.enc(x, pad_mask)).squeeze(-1)

## Component B: Embedding

In [ ]:
class ComponentB(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x)
    return x

# 3. Training

A GPU from Google Colab is preferably used.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

device = "cuda" if torch.cuda.is_available() else "cpu"

y = torch.from_numpy(y_train["label"].to_numpy()).float()

idx_tr, idx_va = train_test_split(np.arange(len(y)), test_size=0.2, random_state=0)

train_loader = DataLoader(
    TensorDataset(channels[idx_tr], y[idx_tr]),
    batch_size=256,
    shuffle=True
)
val_loader = DataLoader(
    TensorDataset(channels[idx_va], y[idx_va]),
    batch_size=1024
)
y_val = y[idx_va]

print(device, len(idx_tr), len(idx_va))


cuda 421658 105415


- `BCEWithLogitsLoss` is used. It combines a sigmoid layer with the Binary Cross-Entropy loss, which is supposed to provide more numerical stability.
- `AdamW` as the optimizer, this is the most performant.

In [ ]:
model = ComponentA(n_channels=2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(20):
    model.train()
    for x, y_batch in train_loader:
        x, y_batch = x.to(device), y_batch.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x), y_batch)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        p = torch.cat([torch.sigmoid(model(x.to(device))).cpu() for x, _ in val_loader])
    acc = ((p.numpy() > 0.5) == y_val.numpy()).mean()
    print(f"epoch {epoch}  acc {acc:.4f}")

epoch 0  acc 0.5152
epoch 1  acc 0.5152
epoch 2  acc 0.5174
epoch 3  acc 0.5167
epoch 4  acc 0.5169
epoch 5  acc 0.5177
epoch 6  acc 0.5176
epoch 7  acc 0.5177
epoch 8  acc 0.5178
epoch 9  acc 0.5175
epoch 10  acc 0.5156
epoch 11  acc 0.5171
epoch 12  acc 0.5161
epoch 13  acc 0.5183
epoch 14  acc 0.5167
epoch 15  acc 0.5193
epoch 16  acc 0.5181
epoch 17  acc 0.5175
epoch 18  acc 0.5182
epoch 19  acc 0.5178


# 4. Evaluation